# A small two-dimensional wildfire simulation

The model follows two quantities in each cell: temperature $T$ (K) and remaining dry fuel $Y$ (1 means unburned). Heat moves between cells by diffusion and with the effective wind. Hot fuel burns, releases heat, and lowers $Y$; heat loss cools the cell. Moisture absorbs heat as it evaporates.

This notebook uses the complete Navas-Montilla solver on a small mesh. It keeps its plots and measurements here; running it creates no files.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from wildfire_model.helpers import burned_area
from wildfire_model.inputs.config import CompleteModelConfiguration
from wildfire_model.inputs.initial_conditions import circular_ignition
from wildfire_model.mesh.mesh_2d import UniformMesh2D
from wildfire_model.numerics.complete_solver_2d import NavasMontillaSolver2D

## Initial state and physical inputs

The domain is 20 m by 20 m, with 40 cells along each direction. A 2 m hot circle starts at $(7,10)$ m. The remaining-fuel fraction `Y` starts at one everywhere. The supplied wind points toward increasing $x$ at 0.25 m/s; the model applies its wind correction to obtain effective flow. We use the S2M moisture model with a dry-basis moisture ratio of 0.1; the remaining physical values are the package defaults.

In [ ]:
mesh = UniformMesh2D(0.0, 20.0, 0.0, 20.0, nx=40, ny=40)
initial = circular_ignition(
    mesh,
    (7.0, 10.0),
    2.0,
    ignition_temperature=670.0,
    ambient_temperature=300.0,
)
configuration = CompleteModelConfiguration(
    wind_x=0.25,
    moisture_model="S2M",
    moisture=0.1,
)
solver = NavasMontillaSolver2D(mesh, configuration)
print(
    f"Cell field shape: {mesh.shape}; initial CFL limit: {solver.maximum_time_step(initial):.3f} s"
)

## Advance the fire

The solver chooses steps below its CFL limit and returns snapshots at 0, 20, 40, and 60 seconds. The small checks below catch invalid shapes or nonphysical output; the printed values are measurements of this run.

In [ ]:
times = np.array([0.0, 20.0, 40.0, 60.0])
result = solver.solve(initial, times)
assert result.temperature.shape == (len(times), *mesh.shape)
assert np.isfinite(result.temperature).all()
assert np.all(result.temperature > 0.0)
assert np.all((result.biomass >= 0.0) & (result.biomass <= 1.0))

print("time (s)   max T (K)   min Y   area with Y < 0.99 (m²)")
for index, time in enumerate(result.times):
    temperature = result.temperature[index]
    biomass = result.biomass[index]
    area = burned_area(biomass, mesh)
    print(f"{time:8.0f}   {temperature.max():9.1f}   {biomass.min():.3f}   {area:8.2f}")

## Read the result

The top row shows temperature; the bottom row shows fuel remaining at the same times. Both use fixed colour scales, so changes between panels are comparable. The hot region extends beyond the original circle before every warm cell has consumed fuel.

In [ ]:
snapshot_indices = (0, 1, 3)
figure, axes = plt.subplots(2, 3, figsize=(12, 6), constrained_layout=True)
extent = (mesh.x_min, mesh.x_max, mesh.y_min, mesh.y_max)
for column, index in enumerate(snapshot_indices):
    heat = axes[0, column].imshow(
        result.temperature[index],
        origin="lower",
        extent=extent,
        vmin=300,
        vmax=1200,
        cmap="inferno",
    )
    fuel = axes[1, column].imshow(
        result.biomass[index],
        origin="lower",
        extent=extent,
        vmin=0.5,
        vmax=1.0,
        cmap="viridis",
    )
    axes[0, column].set_title(f"{result.times[index]:.0f} s")
    axes[1, column].set_xlabel("x (m)")
    for row in (0, 1):
        axes[row, column].set_ylabel("y (m)")
figure.colorbar(heat, ax=axes[0, :], label="Temperature (K)", shrink=0.8)
figure.colorbar(fuel, ax=axes[1, :], label="Remaining fuel (-)", shrink=0.8)
plt.show()

The arrays `result.temperature` and `result.biomass` contain every requested snapshot in `(time, y, x)` order. Change the wind, moisture, ignition, or output times above and rerun the cells to explore another case. This small run illustrates the equations; it is not a quantitative reproduction of a published fire.